In [1]:
from math import pi
import numpy as np
import scipy as sp
from qiskit.quantum_info import Pauli
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.circuit import Parameter
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SciPyOptimizer
from qiskit.primitives import Sampler # new
from qiskit.quantum_info import SparsePauliOp,PauliList
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.primitives import Estimator # new
import time
import pandas as pd
from qiskit.providers.basic_provider import BasicProvider #new
from multiprocessing import Pool
import multiprocessing as mp
from qiskit_algorithms.optimizers import SciPyOptimizer
import scipy as sp
import os
#from sko.SA import SA

In [2]:
def create_ham_str(nqubits):

    # Create a list of terms for the hamiltonian (open boundary conditions)
    # Inputs: nqubits (int), number of qubits
    # Outputs: ham (list), hamiltonian 

    ham = []

    for i in range(nqubits-1):

        term = ''

        for j in range(nqubits-i-2):

            term += 'I'

        for j in range(nqubits-i-2,nqubits-i):

            term += 'Y'  # Choose a Pauli matrix, i.e., X, Y or Z

        for j in range(nqubits-i,nqubits):

            term += 'I'

        ham.append(term)

    return ham



def evaluate_expectation(parameters_values):
    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars = list(value_dict.values())
    expectation_value = estimator.run(ansatz, qubit_op, pars).result().values
    return np.real(expectation_value[0])  # Ensure it returns a scalar

In [ ]:
min_qubits = 5
max_qubits=6
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                backend_options={ # method chosen automatically to match options
                    "coupling_map": coupling_map,
                },
                run_options={"seed": 42, "shots": 1024},
                transpile_options={"seed_transpiler": 42},
            )
    # Example usage
    from mealpy import FloatVar, AOA
    
    problem_dict = {
        "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
        "minmax": "min",
        "obj_func": evaluate_expectation
    }

    model = AOA.OriginalAOA(epoch=1000, pop_size=50, alpha = 5, miu = 0.5, moa_min = 0.2, moa_max = 0.9)
    g_best = model.solve(problem_dict)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

ansatz_num_parameters= 20


2024/07/11 04:26:20 PM, INFO, mealpy.math_based.AOA.OriginalAOA: Solving single objective optimization problem.
2024/07/11 04:26:20 PM, INFO, mealpy.math_based.AOA.OriginalAOA: >>>Problem: P, Epoch: 1, Current best: -1.40625, Global best: -1.40625, Runtime: 0.14367 seconds
2024/07/11 04:26:20 PM, INFO, mealpy.math_based.AOA.OriginalAOA: >>>Problem: P, Epoch: 2, Current best: -1.40625, Global best: -1.40625, Runtime: 0.14983 seconds
2024/07/11 04:26:20 PM, INFO, mealpy.math_based.AOA.OriginalAOA: >>>Problem: P, Epoch: 3, Current best: -1.40625, Global best: -1.40625, Runtime: 0.14779 seconds
2024/07/11 04:26:21 PM, INFO, mealpy.math_based.AOA.OriginalAOA: >>>Problem: P, Epoch: 4, Current best: -1.40625, Global best: -1.40625, Runtime: 0.14770 seconds
2024/07/11 04:26:21 PM, INFO, mealpy.math_based.AOA.OriginalAOA: >>>Problem: P, Epoch: 5, Current best: -1.40625, Global best: -1.40625, Runtime: 0.14835 seconds
2024/07/11 04:26:21 PM, INFO, mealpy.math_based.AOA.OriginalAOA: >>>Problem: P

KeyboardInterrupt: 

In [4]:
from mealpy import FloatVar, CEM

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = CEM.OriginalCEM(epoch=1000, pop_size=50, n_best = 20, alpha = 0.7)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:27:28 PM, INFO, mealpy.math_based.CEM.OriginalCEM: Solving single objective optimization problem.
2024/07/11 04:27:28 PM, INFO, mealpy.math_based.CEM.OriginalCEM: >>>Problem: P, Epoch: 1, Current best: -1.21875, Global best: -1.21875, Runtime: 0.19714 seconds
2024/07/11 04:27:28 PM, INFO, mealpy.math_based.CEM.OriginalCEM: >>>Problem: P, Epoch: 2, Current best: -1.4375, Global best: -1.4375, Runtime: 0.14472 seconds
2024/07/11 04:27:28 PM, INFO, mealpy.math_based.CEM.OriginalCEM: >>>Problem: P, Epoch: 3, Current best: -1.4375, Global best: -1.4375, Runtime: 0.17585 seconds
2024/07/11 04:27:29 PM, INFO, mealpy.math_based.CEM.OriginalCEM: >>>Problem: P, Epoch: 4, Current best: -1.4375, Global best: -1.4375, Runtime: 0.15962 seconds
2024/07/11 04:27:29 PM, INFO, mealpy.math_based.CEM.OriginalCEM: >>>Problem: P, Epoch: 5, Current best: -1.4375, Global best: -1.4375, Runtime: 0.16395 seconds
2024/07/11 04:27:29 PM, INFO, mealpy.math_based.CEM.OriginalCEM: >>>Problem: P, Epoch:

Solution: [ -99.91390389  -99.99256882   99.98917567  -99.97521262  -99.98615583
 -100.         -100.         -100.         -100.         -100.
  100.          100.         -100.          100.          100.
  100.          100.          100.          100.          100.        ], Fitness: -2.15625
Solution: [ -99.91390389  -99.99256882   99.98917567  -99.97521262  -99.98615583
 -100.         -100.         -100.         -100.         -100.
  100.          100.         -100.          100.          100.
  100.          100.          100.          100.          100.        ], Fitness: -2.15625


In [6]:
from mealpy import FloatVar, CGO
objective_function = evaluate_expectation    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = CGO.OriginalCGO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:38:08 PM, INFO, mealpy.math_based.CGO.OriginalCGO: Solving single objective optimization problem.
2024/07/11 04:38:09 PM, INFO, mealpy.math_based.CGO.OriginalCGO: >>>Problem: P, Epoch: 1, Current best: -1.4375, Global best: -1.4375, Runtime: 0.65069 seconds
2024/07/11 04:38:10 PM, INFO, mealpy.math_based.CGO.OriginalCGO: >>>Problem: P, Epoch: 2, Current best: -2.375, Global best: -2.375, Runtime: 0.65010 seconds
2024/07/11 04:38:11 PM, INFO, mealpy.math_based.CGO.OriginalCGO: >>>Problem: P, Epoch: 3, Current best: -2.375, Global best: -2.375, Runtime: 0.67277 seconds
2024/07/11 04:38:11 PM, INFO, mealpy.math_based.CGO.OriginalCGO: >>>Problem: P, Epoch: 4, Current best: -2.46875, Global best: -2.46875, Runtime: 0.68774 seconds
2024/07/11 04:38:12 PM, INFO, mealpy.math_based.CGO.OriginalCGO: >>>Problem: P, Epoch: 5, Current best: -2.5625, Global best: -2.5625, Runtime: 0.52516 seconds
2024/07/11 04:38:12 PM, INFO, mealpy.math_based.CGO.OriginalCGO: >>>Problem: P, Epoch: 6, 

KeyboardInterrupt: 

In [7]:
from mealpy import FloatVar, CircleSA
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = CircleSA.OriginalCircleSA(epoch=1000, pop_size=50, c_factor=0.8)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:40:21 PM, INFO, mealpy.math_based.CircleSA.OriginalCircleSA: Solving single objective optimization problem.
2024/07/11 04:40:22 PM, INFO, mealpy.math_based.CircleSA.OriginalCircleSA: >>>Problem: P, Epoch: 1, Current best: -1.3125, Global best: -1.3125, Runtime: 0.23726 seconds
2024/07/11 04:40:22 PM, INFO, mealpy.math_based.CircleSA.OriginalCircleSA: >>>Problem: P, Epoch: 2, Current best: -1.34375, Global best: -1.34375, Runtime: 0.20616 seconds
2024/07/11 04:40:22 PM, INFO, mealpy.math_based.CircleSA.OriginalCircleSA: >>>Problem: P, Epoch: 3, Current best: -1.46875, Global best: -1.46875, Runtime: 0.20794 seconds
2024/07/11 04:40:22 PM, INFO, mealpy.math_based.CircleSA.OriginalCircleSA: >>>Problem: P, Epoch: 4, Current best: -1.46875, Global best: -1.46875, Runtime: 0.23453 seconds
2024/07/11 04:40:22 PM, INFO, mealpy.math_based.CircleSA.OriginalCircleSA: >>>Problem: P, Epoch: 5, Current best: -1.90625, Global best: -1.90625, Runtime: 0.20838 seconds
2024/07/11 04:40:23 

KeyboardInterrupt: 

In [9]:
from mealpy import FloatVar, GBO
    
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = GBO.OriginalGBO(epoch=1000, pop_size=50, pr = 0.5, beta_min = 0.2, beta_max = 1.2)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:45:01 PM, INFO, mealpy.math_based.GBO.OriginalGBO: Solving single objective optimization problem.
2024/07/11 04:45:01 PM, INFO, mealpy.math_based.GBO.OriginalGBO: >>>Problem: P, Epoch: 1, Current best: -1.46875, Global best: -1.46875, Runtime: 0.21107 seconds
2024/07/11 04:45:02 PM, INFO, mealpy.math_based.GBO.OriginalGBO: >>>Problem: P, Epoch: 2, Current best: -1.46875, Global best: -1.46875, Runtime: 0.21917 seconds
2024/07/11 04:45:02 PM, INFO, mealpy.math_based.GBO.OriginalGBO: >>>Problem: P, Epoch: 3, Current best: -1.53125, Global best: -1.53125, Runtime: 0.22741 seconds
2024/07/11 04:45:02 PM, INFO, mealpy.math_based.GBO.OriginalGBO: >>>Problem: P, Epoch: 4, Current best: -1.53125, Global best: -1.53125, Runtime: 0.21104 seconds
2024/07/11 04:45:03 PM, INFO, mealpy.math_based.GBO.OriginalGBO: >>>Problem: P, Epoch: 5, Current best: -1.53125, Global best: -1.53125, Runtime: 0.38555 seconds
2024/07/11 04:45:03 PM, INFO, mealpy.math_based.GBO.OriginalGBO: >>>Problem: P

KeyboardInterrupt: 

In [10]:
from mealpy import FloatVar, HC
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = HC.OriginalHC(epoch=1000, pop_size=50, neighbour_size = 50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:45:48 PM, INFO, mealpy.math_based.HC.OriginalHC: Solving single objective optimization problem.
2024/07/11 04:45:49 PM, INFO, mealpy.math_based.HC.OriginalHC: >>>Problem: P, Epoch: 1, Current best: -1.46875, Global best: -1.46875, Runtime: 0.19031 seconds
2024/07/11 04:45:49 PM, INFO, mealpy.math_based.HC.OriginalHC: >>>Problem: P, Epoch: 2, Current best: -1.0, Global best: -1.46875, Runtime: 0.17054 seconds
2024/07/11 04:45:49 PM, INFO, mealpy.math_based.HC.OriginalHC: >>>Problem: P, Epoch: 3, Current best: -1.4375, Global best: -1.46875, Runtime: 0.19394 seconds
2024/07/11 04:45:49 PM, INFO, mealpy.math_based.HC.OriginalHC: >>>Problem: P, Epoch: 4, Current best: -0.875, Global best: -1.46875, Runtime: 0.20358 seconds
2024/07/11 04:45:50 PM, INFO, mealpy.math_based.HC.OriginalHC: >>>Problem: P, Epoch: 5, Current best: -2.03125, Global best: -2.03125, Runtime: 0.21555 seconds
2024/07/11 04:45:50 PM, INFO, mealpy.math_based.HC.OriginalHC: >>>Problem: P, Epoch: 6, Current b

KeyboardInterrupt: 

In [11]:
from mealpy import FloatVar, INFO
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = INFO.OriginalINFO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")


2024/07/11 04:47:46 PM, INFO, mealpy.math_based.INFO.OriginalINFO: Solving single objective optimization problem.
c:\Users\petre\psi4conda\envs\psi4conda\Lib\site-packages\mealpy\math_based\INFO.py:108: RuntimeWarning: divide by zero encountered in divide
  z1 = self.pop[idx].solution + sigma * (self.generator.random() * mean_rule) + self.generator.random() * \
c:\Users\petre\psi4conda\envs\psi4conda\Lib\site-packages\mealpy\math_based\INFO.py:119: RuntimeWarning: invalid value encountered in add
  u1 = z1 + mu * np.abs(z1 - z2)      # Eq. (10.1)
2024/07/11 04:47:46 PM, INFO, mealpy.math_based.INFO.OriginalINFO: >>>Problem: P, Epoch: 1, Current best: -1.375, Global best: -1.375, Runtime: 0.19089 seconds
c:\Users\petre\psi4conda\envs\psi4conda\Lib\site-packages\mealpy\math_based\INFO.py:110: RuntimeWarning: divide by zero encountered in divide
  z2 = self.g_best.solution + sigma * (self.generator.random() * mean_rule) + self.generator.random() * \
c:\Users\petre\psi4conda\envs\psi4conda

TypeError: Invalid parameter values, expected Sequence[Sequence[float]].

In [13]:
from mealpy import FloatVar, PSS
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = PSS.OriginalPSS(epoch=1000, pop_size=50, acceptance_rate = 0.8, sampling_method = "LHS")
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:50:27 PM, INFO, mealpy.math_based.PSS.OriginalPSS: Solving single objective optimization problem.
2024/07/11 04:50:28 PM, INFO, mealpy.math_based.PSS.OriginalPSS: >>>Problem: P, Epoch: 1, Current best: -1.375, Global best: -1.375, Runtime: 0.25805 seconds
2024/07/11 04:50:28 PM, INFO, mealpy.math_based.PSS.OriginalPSS: >>>Problem: P, Epoch: 2, Current best: -1.375, Global best: -1.375, Runtime: 0.23394 seconds
2024/07/11 04:50:28 PM, INFO, mealpy.math_based.PSS.OriginalPSS: >>>Problem: P, Epoch: 3, Current best: -1.3125, Global best: -1.375, Runtime: 0.38246 seconds
2024/07/11 04:50:29 PM, INFO, mealpy.math_based.PSS.OriginalPSS: >>>Problem: P, Epoch: 4, Current best: -1.1875, Global best: -1.375, Runtime: 0.24644 seconds
2024/07/11 04:50:29 PM, INFO, mealpy.math_based.PSS.OriginalPSS: >>>Problem: P, Epoch: 5, Current best: -0.9375, Global best: -1.375, Runtime: 0.24403 seconds
2024/07/11 04:50:29 PM, INFO, mealpy.math_based.PSS.OriginalPSS: >>>Problem: P, Epoch: 6, Curre

Solution: [  28.32790878   27.14751186   13.00939312   55.45579115 -100.
   23.68115964    0.97021787    2.81895679   64.70423734  -17.84166688
  -47.98002241   -2.04930593  -40.8525058   -15.00218904   -0.49460171
  -55.58796098  -71.08090693  -86.26930291   39.37295502   60.07254487], Fitness: -2.875
Solution: [  28.32790878   27.14751186   13.00939312   55.45579115 -100.
   23.68115964    0.97021787    2.81895679   64.70423734  -17.84166688
  -47.98002241   -2.04930593  -40.8525058   -15.00218904   -0.49460171
  -55.58796098  -71.08090693  -86.26930291   39.37295502   60.07254487], Fitness: -2.875


In [14]:
from mealpy import FloatVar, RUN
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = RUN.OriginalRUN(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:54:01 PM, INFO, mealpy.math_based.RUN.OriginalRUN: Solving single objective optimization problem.
2024/07/11 04:54:01 PM, INFO, mealpy.math_based.RUN.OriginalRUN: >>>Problem: P, Epoch: 1, Current best: -1.34375, Global best: -1.34375, Runtime: 0.46277 seconds
2024/07/11 04:54:02 PM, INFO, mealpy.math_based.RUN.OriginalRUN: >>>Problem: P, Epoch: 2, Current best: -1.34375, Global best: -1.34375, Runtime: 0.42551 seconds
2024/07/11 04:54:02 PM, INFO, mealpy.math_based.RUN.OriginalRUN: >>>Problem: P, Epoch: 3, Current best: -1.34375, Global best: -1.34375, Runtime: 0.41025 seconds
2024/07/11 04:54:02 PM, INFO, mealpy.math_based.RUN.OriginalRUN: >>>Problem: P, Epoch: 4, Current best: -1.34375, Global best: -1.34375, Runtime: 0.34958 seconds
2024/07/11 04:54:03 PM, INFO, mealpy.math_based.RUN.OriginalRUN: >>>Problem: P, Epoch: 5, Current best: -1.34375, Global best: -1.34375, Runtime: 0.52966 seconds
2024/07/11 04:54:03 PM, INFO, mealpy.math_based.RUN.OriginalRUN: >>>Problem: P

Solution: [-97.88010183 -99.57491868  99.86556661 -40.67584414 -72.89240996
 -45.60673688 -27.19968433 -79.59934215  -6.19295866  66.43142148
  29.83295305  99.9180987   39.189205    46.3752772  -99.09501486
  26.97201609  94.8112839  -98.88560909  36.19794798 -64.52496764], Fitness: -3.96875
Solution: [-97.88010183 -99.57491868  99.86556661 -40.67584414 -72.89240996
 -45.60673688 -27.19968433 -79.59934215  -6.19295866  66.43142148
  29.83295305  99.9180987   39.189205    46.3752772  -99.09501486
  26.97201609  94.8112839  -98.88560909  36.19794798 -64.52496764], Fitness: -3.96875


In [15]:
from mealpy import FloatVar, SCA
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = SCA.DevSCA(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:58:52 PM, INFO, mealpy.math_based.SCA.DevSCA: Solving single objective optimization problem.
2024/07/11 04:58:53 PM, INFO, mealpy.math_based.SCA.DevSCA: >>>Problem: P, Epoch: 1, Current best: -1.28125, Global best: -1.28125, Runtime: 0.16653 seconds
2024/07/11 04:58:53 PM, INFO, mealpy.math_based.SCA.DevSCA: >>>Problem: P, Epoch: 2, Current best: -1.5625, Global best: -1.5625, Runtime: 0.16246 seconds
2024/07/11 04:58:53 PM, INFO, mealpy.math_based.SCA.DevSCA: >>>Problem: P, Epoch: 3, Current best: -1.75, Global best: -1.75, Runtime: 0.16435 seconds
2024/07/11 04:58:53 PM, INFO, mealpy.math_based.SCA.DevSCA: >>>Problem: P, Epoch: 4, Current best: -1.75, Global best: -1.75, Runtime: 0.16629 seconds
2024/07/11 04:58:53 PM, INFO, mealpy.math_based.SCA.DevSCA: >>>Problem: P, Epoch: 5, Current best: -1.75, Global best: -1.75, Runtime: 0.16303 seconds
2024/07/11 04:58:54 PM, INFO, mealpy.math_based.SCA.DevSCA: >>>Problem: P, Epoch: 6, Current best: -1.75, Global best: -1.75, Ru

Solution: [  3.8966062  -98.87941979  11.45806877  98.55785368  21.85850074
  20.28120503  18.26149923  24.7874496   25.73284499  63.94937906
  50.34031854 -98.70069954 -14.26234566  73.30169313 -17.50624246
 -85.30640784  45.79320874 -93.07267452   1.87111324 -99.95621714], Fitness: -3.0
Solution: [  3.8966062  -98.87941979  11.45806877  98.55785368  21.85850074
  20.28120503  18.26149923  24.7874496   25.73284499  63.94937906
  50.34031854 -98.70069954 -14.26234566  73.30169313 -17.50624246
 -85.30640784  45.79320874 -93.07267452   1.87111324 -99.95621714], Fitness: -3.0


In [21]:
from mealpy import FloatVar, SHIO
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = SHIO.OriginalSHIO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 05:09:25 PM, INFO, mealpy.math_based.SHIO.OriginalSHIO: Solving single objective optimization problem.
2024/07/11 05:09:26 PM, INFO, mealpy.math_based.SHIO.OriginalSHIO: >>>Problem: P, Epoch: 1, Current best: -0.96875, Global best: -1.3125, Runtime: 0.16060 seconds
2024/07/11 05:09:26 PM, INFO, mealpy.math_based.SHIO.OriginalSHIO: >>>Problem: P, Epoch: 2, Current best: -0.8125, Global best: -1.3125, Runtime: 0.15680 seconds
2024/07/11 05:09:26 PM, INFO, mealpy.math_based.SHIO.OriginalSHIO: >>>Problem: P, Epoch: 3, Current best: -1.375, Global best: -1.375, Runtime: 0.15880 seconds
2024/07/11 05:09:26 PM, INFO, mealpy.math_based.SHIO.OriginalSHIO: >>>Problem: P, Epoch: 4, Current best: -1.25, Global best: -1.375, Runtime: 0.17391 seconds
2024/07/11 05:09:26 PM, INFO, mealpy.math_based.SHIO.OriginalSHIO: >>>Problem: P, Epoch: 5, Current best: -1.1875, Global best: -1.375, Runtime: 0.14911 seconds
2024/07/11 05:09:26 PM, INFO, mealpy.math_based.SHIO.OriginalSHIO: >>>Problem: P,

Solution: [-1.01537829e+01 -3.89653213e+00  2.15878785e+00 -3.59815204e-01
  1.65280704e+01  9.22338026e-01 -1.88649616e-01 -4.97387605e+01
 -7.27173984e-03  1.10687551e+01 -7.77322981e+00 -7.87300446e+00
 -3.32030254e+01 -1.09290009e+01  2.74465378e+00 -4.72818746e+00
  7.91929533e+00 -3.60686214e+01 -4.68212133e+00  4.73056705e-03], Fitness: -3.96875
Solution: [-1.01537829e+01 -3.89653213e+00  2.15878785e+00 -3.59815204e-01
  1.65280704e+01  9.22338026e-01 -1.88649616e-01 -4.97387605e+01
 -7.27173984e-03  1.10687551e+01 -7.77322981e+00 -7.87300446e+00
 -3.32030254e+01 -1.09290009e+01  2.74465378e+00 -4.72818746e+00
  7.91929533e+00 -3.60686214e+01 -4.68212133e+00  4.73056705e-03], Fitness: -3.96875


In [20]:
from mealpy import FloatVar, TS
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = TS.OriginalTS(epoch=1000, pop_size=200, tabu_size = 5, neighbour_size = 20, perturbation_scale = 0.05)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 05:08:32 PM, INFO, mealpy.math_based.TS.OriginalTS: Solving single objective optimization problem.
2024/07/11 05:08:33 PM, INFO, mealpy.math_based.TS.OriginalTS: >>>Problem: P, Epoch: 1, Current best: -2.03125, Global best: -2.03125, Runtime: 0.10491 seconds
2024/07/11 05:08:33 PM, INFO, mealpy.math_based.TS.OriginalTS: >>>Problem: P, Epoch: 2, Current best: -2.03125, Global best: -2.03125, Runtime: 0.08324 seconds
2024/07/11 05:08:33 PM, INFO, mealpy.math_based.TS.OriginalTS: >>>Problem: P, Epoch: 3, Current best: -2.03125, Global best: -2.03125, Runtime: 0.20498 seconds
2024/07/11 05:08:33 PM, INFO, mealpy.math_based.TS.OriginalTS: >>>Problem: P, Epoch: 4, Current best: -2.03125, Global best: -2.03125, Runtime: 0.09237 seconds
2024/07/11 05:08:34 PM, INFO, mealpy.math_based.TS.OriginalTS: >>>Problem: P, Epoch: 5, Current best: -2.03125, Global best: -2.03125, Runtime: 0.14641 seconds
2024/07/11 05:08:34 PM, INFO, mealpy.math_based.TS.OriginalTS: >>>Problem: P, Epoch: 6, Cu

KeyboardInterrupt: 

In [19]:
from mealpy import FloatVar, HS
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = HS.DevHS(epoch=1000, pop_size=50, c_r = 0.95, pa_r = 0.05)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 05:07:48 PM, INFO, mealpy.music_based.HS.DevHS: Solving single objective optimization problem.
2024/07/11 05:07:49 PM, INFO, mealpy.music_based.HS.DevHS: >>>Problem: P, Epoch: 1, Current best: -1.59375, Global best: -1.59375, Runtime: 0.20899 seconds
2024/07/11 05:07:49 PM, INFO, mealpy.music_based.HS.DevHS: >>>Problem: P, Epoch: 2, Current best: -1.8125, Global best: -1.8125, Runtime: 0.20293 seconds
2024/07/11 05:07:49 PM, INFO, mealpy.music_based.HS.DevHS: >>>Problem: P, Epoch: 3, Current best: -1.9375, Global best: -1.9375, Runtime: 0.19110 seconds
2024/07/11 05:07:49 PM, INFO, mealpy.music_based.HS.DevHS: >>>Problem: P, Epoch: 4, Current best: -1.9375, Global best: -1.9375, Runtime: 0.34063 seconds
2024/07/11 05:07:50 PM, INFO, mealpy.music_based.HS.DevHS: >>>Problem: P, Epoch: 5, Current best: -2.03125, Global best: -2.03125, Runtime: 0.28806 seconds
2024/07/11 05:07:50 PM, INFO, mealpy.music_based.HS.DevHS: >>>Problem: P, Epoch: 6, Current best: -2.59375, Global best:

KeyboardInterrupt: 

In [23]:
from mealpy import FloatVar, AEO
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = AEO.OriginalAEO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 05:15:25 PM, INFO, mealpy.system_based.AEO.OriginalAEO: Solving single objective optimization problem.
2024/07/11 05:15:25 PM, INFO, mealpy.system_based.AEO.OriginalAEO: >>>Problem: P, Epoch: 1, Current best: -1.75, Global best: -1.75, Runtime: 0.33688 seconds
2024/07/11 05:15:26 PM, INFO, mealpy.system_based.AEO.OriginalAEO: >>>Problem: P, Epoch: 2, Current best: -1.75, Global best: -1.75, Runtime: 0.34819 seconds
2024/07/11 05:15:26 PM, INFO, mealpy.system_based.AEO.OriginalAEO: >>>Problem: P, Epoch: 3, Current best: -1.75, Global best: -1.75, Runtime: 0.30113 seconds
2024/07/11 05:15:26 PM, INFO, mealpy.system_based.AEO.OriginalAEO: >>>Problem: P, Epoch: 4, Current best: -1.75, Global best: -1.75, Runtime: 0.34958 seconds
2024/07/11 05:15:27 PM, INFO, mealpy.system_based.AEO.OriginalAEO: >>>Problem: P, Epoch: 5, Current best: -1.75, Global best: -1.75, Runtime: 0.31076 seconds
2024/07/11 05:15:27 PM, INFO, mealpy.system_based.AEO.OriginalAEO: >>>Problem: P, Epoch: 6, Curr

Solution: [ 98.0822213   74.97613869  99.54168909 -99.35709237  36.74952771
 -74.68839483 -24.96334698 -10.64323228  98.89788041  98.5042962
 -55.92661976 -98.47591218 -55.23530477  99.35000976  45.50787013
  98.28870344 -99.14645105 -99.009978   -97.93932458 -98.97317281], Fitness: -3.84375
Solution: [ 98.0822213   74.97613869  99.54168909 -99.35709237  36.74952771
 -74.68839483 -24.96334698 -10.64323228  98.89788041  98.5042962
 -55.92661976 -98.47591218 -55.23530477  99.35000976  45.50787013
  98.28870344 -99.14645105 -99.009978   -97.93932458 -98.97317281], Fitness: -3.84375


In [24]:
from mealpy import FloatVar, GCO
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = GCO.DevGCO(epoch=1000, pop_size=50, cr = 0.7, wf = 1.25)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 05:20:59 PM, INFO, mealpy.system_based.GCO.DevGCO: Solving single objective optimization problem.
2024/07/11 05:20:59 PM, INFO, mealpy.system_based.GCO.DevGCO: >>>Problem: P, Epoch: 1, Current best: -1.46875, Global best: -1.46875, Runtime: 0.19691 seconds
2024/07/11 05:21:00 PM, INFO, mealpy.system_based.GCO.DevGCO: >>>Problem: P, Epoch: 2, Current best: -1.46875, Global best: -1.46875, Runtime: 0.29937 seconds
2024/07/11 05:21:00 PM, INFO, mealpy.system_based.GCO.DevGCO: >>>Problem: P, Epoch: 3, Current best: -1.46875, Global best: -1.46875, Runtime: 0.40177 seconds
2024/07/11 05:21:00 PM, INFO, mealpy.system_based.GCO.DevGCO: >>>Problem: P, Epoch: 4, Current best: -1.46875, Global best: -1.46875, Runtime: 0.29649 seconds
2024/07/11 05:21:01 PM, INFO, mealpy.system_based.GCO.DevGCO: >>>Problem: P, Epoch: 5, Current best: -1.46875, Global best: -1.46875, Runtime: 0.23925 seconds
2024/07/11 05:21:01 PM, INFO, mealpy.system_based.GCO.DevGCO: >>>Problem: P, Epoch: 6, Current b

KeyboardInterrupt: 

In [27]:
from mealpy import FloatVar, WCA
    
problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = WCA.OriginalWCA(epoch=1000, pop_size=50, nsr = 4, wc = 2.0, dmax = 1e-6)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 05:27:29 PM, INFO, mealpy.system_based.WCA.OriginalWCA: Solving single objective optimization problem.
2024/07/11 05:27:30 PM, INFO, mealpy.system_based.WCA.OriginalWCA: >>>Problem: P, Epoch: 1, Current best: -1.78125, Global best: -1.78125, Runtime: 0.26526 seconds
2024/07/11 05:27:30 PM, INFO, mealpy.system_based.WCA.OriginalWCA: >>>Problem: P, Epoch: 2, Current best: -1.78125, Global best: -1.78125, Runtime: 0.37724 seconds
2024/07/11 05:27:30 PM, INFO, mealpy.system_based.WCA.OriginalWCA: >>>Problem: P, Epoch: 3, Current best: -1.78125, Global best: -1.78125, Runtime: 0.26807 seconds
2024/07/11 05:27:31 PM, INFO, mealpy.system_based.WCA.OriginalWCA: >>>Problem: P, Epoch: 4, Current best: -1.78125, Global best: -1.78125, Runtime: 0.28018 seconds
2024/07/11 05:27:31 PM, INFO, mealpy.system_based.WCA.OriginalWCA: >>>Problem: P, Epoch: 5, Current best: -1.78125, Global best: -1.78125, Runtime: 0.24710 seconds
2024/07/11 05:27:31 PM, INFO, mealpy.system_based.WCA.OriginalWCA:

KeyboardInterrupt: 